# S6_1 — Statistical Significance Testing

**Leaf-JEPA IRP** | Stage 6 — Analysis and Interpretation

**Purpose:** Convert all pairwise performance comparisons into rigorous statistical claims.
Every number in Chapter 4 that compares two methods must be backed by a p-value and effect size.

**Outputs:**
- `significance_matrix.csv` — all pairwise p-values
- `effect_sizes.csv` — Cohen's d for all comparisons
- `significance_heatmap.png` — visual summary
- `bootstrap_ci.json` — confidence intervals for each method

**Research Questions Served:** All — this is the statistical foundation for every RQ answer.


## Initialization

In [1]:
import sys, json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))


from stage6_analysis_and_interpretation.config_stage6 import *
from stage6_analysis_and_interpretation.analysis_utils import (
    set_seed, load_json, save_json, load_baseline_results,
    collect_macro_f1_per_seed, pairwise_significance_matrix,
    plot_significance_matrix, cohens_d_paired, effect_size_label,
    bootstrap_ci, print_section, paired_ttest
)

set_seed(RANDOM_SEED)
STAGE6_FIGURES.mkdir(parents=True, exist_ok=True)
STAGE6_TABLES.mkdir(parents=True, exist_ok=True)
STAGE6_DATA.mkdir(parents=True, exist_ok=True)


## Initialization

In [ ]:
print_section("Loading Results")

# Baselines
baseline_results = load_baseline_results(BASELINES_DIR, BASELINE_IDS)
print(f"  Loaded {len(baseline_results)} baseline results")

# PEFT — adapt this loading to your actual JSON format
peft_results_raw = {}
s1_path = PEFT_RESULTS_DIR / "S1_method_comparison_summary.json"
if s1_path.exists():
    s1_data = load_json(s1_path)
    # Attempt to parse per-method, per-seed results
    # Adjust keys based on your actual Stage 5 output format
    if isinstance(s1_data, dict):
        for method_name, method_data in s1_data.items():
            if isinstance(method_data, dict):
                peft_results_raw[method_name] = method_data
    print(f"  Loaded {len(peft_results_raw)} PEFT method results")
else:
    print("  \u26a0\ufe0f S1 comparison summary not found — loading individual files")
    for f in PEFT_RESULTS_DIR.glob("*_results.json"):
        key = f.stem.replace("_results", "")
        peft_results_raw[key] = load_json(f)
    print(f"  Loaded {len(peft_results_raw)} PEFT method results from individual files")


## Extract macro-F1 (per seed)

In [ ]:
print_section("Extracting Per-Seed Macro-F1")

method_scores = {}

# Baselines
for bid, data in baseline_results.items():
    scores = collect_macro_f1_per_seed(data)
    if scores is not None:
        method_scores[bid] = scores
        print(f"  {bid}: {scores} (mean={scores.mean():.4f} \u00b1 {scores.std():.4f})")
    else:
        print(f"  \u26a0\ufe0f {bid}: could not extract per-seed scores")

# PEFT methods
for method_name, data in peft_results_raw.items():
    scores = collect_macro_f1_per_seed(data)
    if scores is not None:
        method_scores[method_name] = scores
        print(f"  {method_name}: {scores} (mean={scores.mean():.4f} \u00b1 {scores.std():.4f})")

print(f"\n  Total methods with per-seed data: {len(method_scores)}")
method_names = list(method_scores.keys())


## Significance test (Pairwise)

In [ ]:
print_section("Pairwise Significance Testing")

# Wilcoxon
print("  Running Wilcoxon signed-rank tests...")
p_wilcoxon, d_matrix = pairwise_significance_matrix(method_scores, method_names, test="wilcoxon")

# Paired t-test (for comparison)
print("  Running paired t-tests...")
p_ttest, _ = pairwise_significance_matrix(method_scores, method_names, test="ttest")

# Save
p_wilcoxon.to_csv(STAGE6_TABLES / "significance_matrix.csv")
d_matrix.to_csv(STAGE6_TABLES / "effect_sizes.csv")
p_ttest.to_csv(STAGE6_TABLES / "significance_matrix_ttest.csv")

print(f"\n  Significant pairs (Wilcoxon, alpha={SIGNIFICANCE_ALPHA}):")
for i in range(len(method_names)):
    for j in range(i+1, len(method_names)):
        p = p_wilcoxon.iloc[i, j]
        d = d_matrix.iloc[i, j]
        if p < SIGNIFICANCE_ALPHA:
            label = effect_size_label(d)
            print(f"    {method_names[i]} vs {method_names[j]}: p={p:.4f}, d={d:.3f} ({label})")

# Key comparison: B3 vs B5 (RQ1)
if "B3" in method_scores and "B5" in method_scores:
    print("\n  === RQ1 KEY RESULT: B3 vs B5 ===")
    _, p_rq1 = paired_ttest(method_scores["B3"], method_scores["B5"])
    d_rq1 = cohens_d_paired(method_scores["B3"], method_scores["B5"])
    delta = method_scores["B5"].mean() - method_scores["B3"].mean()
    print(f"  Delta (B5 - B3): {delta:+.4f}")
    print(f"  p-value (t-test): {p_rq1:.4f}")
    print(f"  Cohen's d: {d_rq1:.3f} ({effect_size_label(d_rq1)})")
    print(f"  Interpretation: {'Significant' if p_rq1 < SIGNIFICANCE_ALPHA else 'Not significant at n=3'}")


## Heatmap

In [ ]:
# Plot significance heatmap
print_section("Generating Significance Heatmap")

plot_significance_matrix(
    p_wilcoxon, d_matrix,
    STAGE6_FIGURES / "significance_heatmap.png",
    alpha=SIGNIFICANCE_ALPHA
)

# Bootstrap CIs for each method
print("\nBootstrap 95% CIs:")
ci_results = {}
for name, scores in method_scores.items():
    lo, hi = bootstrap_ci(scores, n_bootstrap=10000)
    ci_results[name] = {"mean": float(scores.mean()), "ci_lower": lo, "ci_upper": hi}
    print(f"  {name}: {scores.mean():.4f} [{lo:.4f}, {hi:.4f}]")

save_json(ci_results, STAGE6_DATA / "bootstrap_confidence_intervals.json")

print("\n\u2705 S6_1 complete.")


## Critical Analysis

### Statistical Power Limitation
With only n=3 seeds, statistical power is inherently low. Many real differences will not reach
p < 0.05 — this is expected and must be stated explicitly in Chapter 4. Report effect sizes
(Cohen's d) alongside p-values: a large effect size with a non-significant p-value is
**practically significant but not statistically confirmable at n=3**.

### Dissertation Write-Up
For each RQ answer in Chapter 5, cite the specific p-value and effect size:
- "Domain pretraining improved linear probe macro-F1 by X.XX percentage points (Cohen's d = Y.YY, p = Z.ZZ, Wilcoxon signed-rank test, n=3)."
- If p > 0.05: "While the improvement did not reach statistical significance at n=3 (p = Z.ZZ), the effect size was [large/medium] (d = Y.YY), suggesting practical significance that would likely be confirmed with additional seeds."

### Multiple Comparisons
With K methods, there are K(K-1)/2 pairwise comparisons. Consider Bonferroni correction
(divide alpha by number of comparisons) or report uncorrected p-values with an explicit
note about the multiple comparison issue. For an IRP, stating the limitation is sufficient;
formal correction is a bonus.
